In [0]:
from pyspark.sql.functions import lit, current_timestamp
import datetime
import traceback

# ============================================================================
# DIAGNOSTIC LOGGING HELPERS
# ============================================================================
def log_state(step_name: str, fqn: str = None):
    """Log current Spark context state with timestamp, catalog, and database"""
    ts = datetime.datetime.now().isoformat()
    catalog = spark.catalog.currentCatalog()
    database = spark.catalog.currentDatabase()
    fqn_str = f" | FQN: {fqn}" if fqn else ""
    print(f"[{ts}] {step_name} | Catalog: {catalog} | Database: {database}{fqn_str}")

def check_table_visibility(fqn: str, step_name: str):
    """Check if table exists and log detailed metadata"""
    try:
        exists = spark.catalog.tableExists(fqn)
        print(f"  └─ [{step_name}] Table exists check: {exists}")
        if exists:
            desc_df = spark.sql(f"DESCRIBE EXTENDED {fqn}")
            desc_output = desc_df.collect()
            print(f"  └─ [{step_name}] DESCRIBE EXTENDED output (first 10 rows):")
            for row in desc_output[:10]:
                print(f"      {row}")
        return exists
    except Exception as e:
        print(f"  └─ [ERROR in {step_name}] Visibility check failed: {str(e)}")
        print(f"      Traceback: {traceback.format_exc()}")
        return False

# ============================================================================
# MAIN FUNCTION
# ============================================================================
def merge_into_stage(source_df, primary_key, source_table, target_table, is_merge_enabled='N'):
    """
    Merges data from a source DataFrame into a Delta target table using audit columns.

    This function performs the following steps:
    1. Adds audit columns (`is_deleted`, `row_inserted_time`, `row_updated_time`) to the source data.
    2. Identifies new or updated records by comparing with the source data with the target table.
    3. Performs an upsert (MERGE) into the target Delta table.
    4. Identifies deleted rows (present in target but not in source) and:
        - Inserts them into a deleted rows audit table. Expects the deleted_rows table to be present.
        - Physically deletes them from the target table.

    Parameters:
        source_df (DataFrame): Dataframe with curated source data that is ready to be pushed to target.
        primary_key (str): The column used as a unique identifier for matching records. It can take any number of columns separated by a comma.
        source_table (str): The source table name in 'catalog.schema.table' format.
        target_table (str): The target table name in 'catalog.schema.table' format.

    Raises:
        Exception: If any error occurs during processing.
    """

    try:
        # DIAGNOSTIC: Log entry point and confirm fully-qualified names
        log_state("FUNCTION ENTRY", source_table)
        log_state("FUNCTION ENTRY", target_table)
        print(f"  └─ is_merge_enabled: {is_merge_enabled}")
        print(f"  └─ primary_key: {primary_key}")

        # DIAGNOSTIC: Confirm current user/identity (important for SPN debugging)
        try:
            current_user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
            print(f"  └─ Current user (SPN or interactive): {current_user}")
        except Exception as e:
            print(f"  └─ [WARNING] Could not retrieve current_user(): {e}")

        if is_merge_enabled == 'N':
            # ====================================================================
            # OVERWRITE MODE
            #====================================================================
            log_state("OVERWRITE MODE - BEFORE WRITE", target_table)
            loggerObject.info(f"Overwrite for table {target_table} started")

            source_df.write.format("delta") \
                .mode("overwrite") \
                .option("mergeSchema", "true") \
                .saveAsTable(target_table)

            log_state("OVERWRITE MODE - AFTER WRITE", target_table)
            check_table_visibility(target_table, "POST-OVERWRITE")
            loggerObject.info(f"Overwrite for table {target_table} completed")

        else:
            # ====================================================================
            # MERGE MODE (the complex path)
            # ====================================================================
            log_state("MERGE MODE - START", target_table)

            # Step 1: Get current timestamp once
            ts = spark.sql("SELECT current_timestamp() AS now").collect()[0]["now"]
            ts_str = ts.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
            print(f"  └─ Timestamp captured: {ts_str}")

            # Create source temp view for EXCEPT logic based on source_df (audit columns are not added yet)
            table_name = target_table.split(".")[-1]  # Extract table name from FQN
            source_view = f"{table_name}_view"

            log_state("CREATING SOURCE TEMP VIEW", source_view)
            print(f"  └─ [WARNING] Temp view created: {source_view} — will NOT persist across notebook boundaries")
            source_df.createOrReplaceTempView(source_view)

            # Verify temp view was created
            try:
                view_exists = spark.catalog.tableExists(source_view)
                print(f"  └─ Source temp view exists check: {view_exists}")
            except Exception as e:
                print(f"  └─ [ERROR] Could not verify source temp view: {e}")

            # ====================================================================
            # TABLE EXISTENCE & AUDIT COLUMN CHECK
            # ====================================================================
            log_state("CHECKING TARGET TABLE EXISTENCE", target_table)
            try:
                # Try to access the table to confirm it exists
                print(f"  └─ Attempting to read {target_table}...")
                spark.table(target_table)
                print(f"  └─ Target table {target_table} exists; proceeding with merge.")
            except Exception as e:
                # Table does not exist — create it and add audit columns
                print(f"  └─ [ERROR] Target table {target_table} does not exist: {e}")
                print(f"  └─ This is a critical error — target table must exist before merge.")
                raise

            # Try to add audit columns if they don't exist
            try:
                log_state("ADDING AUDIT COLUMNS (IF MISSING)", target_table)
                spark.sql(f"""
                    ALTER TABLE {target_table} ADD COLUMNS (
                        is_deleted INT,
                        row_inserted_time TIMESTAMP,
                        row_updated_time TIMESTAMP
                    )
                """)
                print(f"  └─ Audit columns added (or already existed).")
            except Exception as e:
                print(f"  └─ [INFO] Audit columns may already exist: {e}")

            check_table_visibility(target_table, "POST-AUDIT-COLUMNS")

            # ====================================================================
            # IDENTIFY COMMON COLUMNS & PREPARE MERGE
            # ====================================================================
            log_state("ANALYZING TARGET TABLE SCHEMA", target_table)
            target_table_columns = [col.lower() for col in spark.table(target_table).columns]
            print(f"  └─ Target table columns (lowercase): {target_table_columns}")

            if "dw_insert_dt" in target_table_columns:
                target_columns = spark.table(target_table).drop(
                    "DW_Insert_Dt", "is_deleted", "row_inserted_time", "row_updated_time"
                ).columns
            else:
                target_columns = spark.table(target_table).drop(
                    "is_deleted", "row_inserted_time", "row_updated_time"
                ).columns

            target_columns_delete = spark.table(target_table).columns
            common_columns = [col for col in target_columns if col in source_df.columns]
            common_columns_delete = [col for col in target_columns_delete if col in source_df.columns]
            common_columns_list = ", ".join([f"`{col}`" for col in common_columns])
            common_columns_list_for_deletes = ", ".join([f"t.`{col}`" for col in common_columns_delete])

            print(f"  └─ Common columns for merge: {common_columns}")
            print(f"  └─ Common columns for delete sync: {common_columns_delete}")

            # ====================================================================
            # IDENTIFY NEW & UPDATED ROWS (MERGE CANDIDATES)
            # ====================================================================
            log_state("IDENTIFYING NEW/UPDATED ROWS", source_view)
            from pyspark.sql.functions import col, concat_ws, trim, coalesce, md5

            merge_view = f"{table_name}_merge_view"
            print(f"  └─ [WARNING] Creating temp merge view: {merge_view} — will NOT persist across notebook boundaries")

            source_view_df = spark.sql(f"SELECT {common_columns_list} FROM {source_view}")
            target_table_df = spark.sql(f"SELECT {common_columns_list} FROM {target_table}")

            print(f"  └─ Source view row count: {source_view_df.count()}")
            print(f"  └─ Target table row count: {target_table_df.count()}")

            # Create 'mdf_column' column for both DataFrames: cast to string, trim, fillna('*~*'), then concat
            def build_mdf_column(df):
                cols = [trim(coalesce(col(c).cast("string"), lit("*~*"))) for c in df.columns]
                return df.withColumn("mdf_hash_column", md5(concat_ws("||", *cols)))

            source_view_df = build_mdf_column(source_view_df)
            target_table_df = build_mdf_column(target_table_df)

            merge_df = source_view_df.join(
                target_table_df,
                source_view_df.mdf_hash_column == target_table_df.mdf_hash_column,
                how="left"
            ).filter(target_table_df.mdf_hash_column.isNull()).select([source_view_df[c] for c in common_columns])

            merge_row_count = merge_df.count()
            print(f"  └─ Rows identified for merge (new/updated): {merge_row_count}")

            # ====================================================================
            # ADD AUDIT FIELDS TO MERGE CANDIDATES
            # ====================================================================
            log_state("ADDING AUDIT FIELDS TO MERGE CANDIDATES", merge_view)

            if "dw_insert_dt" in target_table_columns:
                merge_df_with_audit = (
                    merge_df
                    .withColumn('DW_Insert_Dt', current_timestamp())
                    .withColumn('is_deleted', lit(0))
                    .withColumn('row_inserted_time', current_timestamp())
                    .withColumn('row_updated_time', current_timestamp())
                )
            else:
                merge_df_with_audit = (
                    merge_df
                    .withColumn('is_deleted', lit(0))
                    .withColumn('row_inserted_time', current_timestamp())
                    .withColumn('row_updated_time', current_timestamp())
                )

            log_state("CREATING MERGE TEMP VIEW", merge_view)
            print(f"  └─ [WARNING] Creating temp view: {merge_view} — will NOT persist across notebook boundaries")
            merge_df_with_audit.createOrReplaceTempView(merge_view)

            # Verify merge view was created
            try:
                merge_view_exists = spark.catalog.tableExists(merge_view)
                print(f"  └─ Merge temp view exists check: {merge_view_exists}")
            except Exception as e:
                print(f"  └─ [ERROR] Could not verify merge temp view: {e}")

            # Get columns for insert logic
            insert_columns = merge_df_with_audit.columns
            insert_columns_formatted = [f"`{col}`" for col in merge_df_with_audit.columns]
            insert_values = [f"source.`{col}`" for col in insert_columns]

            # ====================================================================
            # BUILD MERGE STATEMENT (MULTIPLE PRIMARY KEYS)
            # ====================================================================
            log_state("BUILDING MERGE STATEMENT", target_table)
            loggerObject.info(f"Handle multiple primary key columns + audit columns.")

            pk_cols = [col.strip() for col in primary_key.split(",")]
            print(f"  └─ Primary key columns: {pk_cols}")

            if "dw_insert_dt" in target_table_columns:
                excluded = pk_cols + ['DW_Insert_Dt', 'is_deleted', 'row_inserted_time', 'row_updated_time']
            else:
                excluded = pk_cols + ['is_deleted', 'row_inserted_time', 'row_updated_time']

            on_clause = " AND ".join([f"trim(upper(source.`{col}`)) = trim(upper(target.`{col}`))" for col in pk_cols])
            updatable_columns = [col for col in insert_columns if col not in excluded]

            update_set_clause = ",\n  ".join(
                [f"target.`{col}` = source.`{col}`" for col in updatable_columns] + [
                    "target.is_deleted = 0",
                    f"target.row_updated_time = cast('{ts_str}' AS TIMESTAMP)"
                ]
            )

            loggerObject.info(f"update_set_clause: {update_set_clause}")
            print(f"  └─ ON clause: {on_clause}")
            print(f"  └─ Updatable columns (excluding PK & audit): {updatable_columns}")

            # ====================================================================
            # EXECUTE MERGE INTO TARGET TABLE
            # ====================================================================
            log_state("BEFORE MERGE", target_table)
            print(f"  └─ Merging {merge_view} into {target_table}...")

            loggerObject.info(f"Merging {merge_view} into {target_table}.")

            merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {merge_view} AS source
            ON {on_clause}
            WHEN MATCHED THEN
            UPDATE SET
            {update_set_clause}
            WHEN NOT MATCHED THEN
            INSERT ({", ".join(insert_columns_formatted)})
            VALUES ({", ".join(insert_values)})
            """

            print(f"  └─ MERGE SQL:\n{merge_sql}\n")

            try:
                audit_df = spark.sql(merge_sql)
                log_state("AFTER MERGE - SUCCESS", target_table)

                updated_count = audit_df.select("num_updated_rows").collect()[0][0]
                inserted_count = audit_df.select("num_inserted_rows").collect()[0][0]
                print(f"  └─ updated: {updated_count}, Rows inserted: {inserted_count}")
                loggerObject.info(f"Rows updated: {updated_count}, Rows inserted: {inserted_count}")
                loggerObject.info(f"***************************Merge Operation Completed******************************")

                check_table_visibility(target_table, "POST-MERGE")

            except Exception as merge_error:
                log_state("AFTER MERGE - FAILED", target_table)
                print(f"  └─ [ERROR] MERGE operation failed!")
                print(f"  └─ Error message: {str(merge_error)}")
                print(f"  └─ Full traceback:\n{traceback.format_exc()}")

                # Log current user/identity at time of error
                try:
                    error_user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
                    print(f"  └─ Current user at error time: {error_user}")
                except Exception as user_error:
                    print(f"  └─ [WARNING] Could not retrieve current_user() at error time: {user_error}")

                raise

            # ====================================================================
            # DELETE HANDLING: IDENTIFY ROWS IN TARGET BUT NOT IN SOURCE
            # ====================================================================
            log_state("DELETE HANDLING - START", target_table)
            loggerObject.info(f"Delete Handling - Identify deleted rows - that are in target but not in source")

            delete_on_clause = " AND ".join([f"trim(upper(t.`{col}`)) = trim(upper(s.`{col}`))" for col in pk_cols])

            deleted_df_without_all_columns = spark.sql(f"""
                SELECT {common_columns_list_for_deletes}, 1 AS is_deleted, t.row_inserted_time
                FROM {target_table} t
                LEFT ANTI JOIN {source_view} s
                ON {delete_on_clause}
            """)
            deleted_df = deleted_df_without_all_columns.withColumn('row_updated_time', lit(ts))

            deleted_row_count = deleted_df.count()
            print(f"  └─ Rows identified for deletion (in target, not in source): {deleted_row_count}")

            if deleted_row_count == 0:
                loggerObject.info(f"No deleted rows found for {target_table}. Skipping deletion sync.")
                print(f"  └─ No deleted rows found. Skipping deletion sync.")
            else:
                # ================================================================
                # CREATE DELETED ROWS TEMP VIEW & AUDIT TABLE
                # ================================================================
                deleted_rows_view = f"{table_name}_deleted_rows_view"
                log_state("CREATING DELETED ROWS TEMP VIEW", deleted_rows_view)
                print(f"  └─ [WARNING] Creating temp view: {deleted_rows_view} — will NOT persist across notebook boundaries")
                deleted_df.createOrReplaceTempView(deleted_rows_view)

                # Verify deleted rows view was created
                try:
                    deleted_view_exists = spark.catalog.tableExists(deleted_rows_view)
                    print(f"  └─ Deleted rows temp view exists check: {deleted_view_exists}")
                except Exception as e:
                    print(f"  └─ [ERROR] Could not verify deleted rows temp view: {e}")

                loggerObject.info(f"Deleting rows from {target_table}.")
                print(f"  └─ Processing {deleted_row_count} deleted rows...")

                # Create deleted_rows audit table
                abstract_target_table_name = target_table.split(".")[-1]
                deleted_rows_table = f"{target_table}_deleted_rows"

                log_state("CREATING DELETED ROWS AUDIT TABLE", deleted_rows_table)

                try:
                    tagret_table_location = spark.sql(f"DESCRIBE DETAIL {target_table}").select("location").collect()[0]["location"]
                    delete_table_location = tagret_table_location.replace(abstract_target_table_name, f"{abstract_target_table_name}_deleted_rows")

                    print(f"  └─ Target table location: {tagret_table_location}")
                    print(f"  └─ Deleted rows table location: {delete_table_location}")

                    spark.sql(f"""
                        CREATE TABLE IF NOT EXISTS {deleted_rows_table}
                        USING DELTA
                        LOCATION '{delete_table_location}'
                        AS SELECT * FROM {target_table}
                        WHERE 1 = 0
                    """)

                    log_state("AFTER CREATING DELETED ROWS TABLE", deleted_rows_table)
                    check_table_visibility(deleted_rows_table, "POST-CREATE-DELETED-TABLE")

                except Exception as create_deleted_table_error:
                    print(f"  └─ [ERROR] Failed to create deleted rows table: {str(create_deleted_table_error)}")
                    print(f"  └─ Traceback: {traceback.format_exc()}")
                    raise

                # ================================================================
                # INSERT DELETED ROWS INTO AUDIT TABLE
                # ================================================================
                log_state("BEFORE INSERT INTO DELETED ROWS TABLE", deleted_rows_table)
                loggerObject.info(f"Inserting deleted rows into {deleted_rows_table}.")
                print(f"  └─ Inserting {deleted_row_count} rows into {deleted_rows_table}...")

                merge_on_deleted = " AND ".join([f"trim(upper(t.`{col}`)) = trim(upper(d.`{col}`))" for col in pk_cols])

                try:
                    spark.sql(f"""
                        MERGE INTO {deleted_rows_table} t
                        USING {deleted_rows_view} d
                        ON {merge_on_deleted}
                        WHEN NOT MATCHED THEN
                        INSERT *
                    """)

                    log_state("AFTER INSERT INTO DELETED ROWS TABLE - SUCCESS", deleted_rows_table)
                    check_table_visibility(deleted_rows_table, "POST-INSERT-DELETED-ROWS")

                except Exception as insert_deleted_error:
                    log_state("AFTER INSERT INTO DELETED ROWS TABLE - FAILED", deleted_rows_table)
                    print(f"  └─ [ERROR] Failed to insert into deleted rows table: {str(insert_deleted_error)}")
                    print(f"  └─ Traceback: {traceback.format_exc()}")

                    try:
                        error_user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
                        print(f"  └─ Current user at error time: {error_user}")
                    except Exception as user_error:
                        print(f"  └─ [WARNING] Could not retrieve current_user(): {user_error}")

                    raise

                # ================================================================
                # DELETE DELETED ROWS FROM TARGET TABLE
                # ================================================================
                log_state("BEFORE DELETE FROM TARGET TABLE", target_table)
                loggerObject.info(f"Deleting deleted rows from {target_table}.")
                print(f"  └─ Deleting {deleted_row_count} rows from {target_table}...")

                delete_merge_on = " AND ".join([f"trim(upper(t.`{col}`)) = trim(upper(d.`{col}`))" for col in pk_cols])

                try:
                    spark.sql(f"""
                        MERGE INTO {target_table} t
                        USING {deleted_rows_view} d
                        ON {delete_merge_on}
                        WHEN MATCHED THEN
                        DELETE
                    """)

                    log_state("AFTER DELETE FROM TARGET TABLE - SUCCESS", target_table)
                    check_table_visibility(target_table, "POST-DELETE-FROM-TARGET")

                except Exception as delete_target_error:
                    log_state("AFTER DELETE FROM TARGET TABLE - FAILED", target_table)
                    print(f"  └─ [ERROR] Failed to delete from target table: {str(delete_target_error)}")
                    print(f"  └─ Traceback: {traceback.format_exc()}")

                    try:
                        error_user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
                        print(f"  └─ Current user at error time: {error_user}")
                    except Exception as user_error:
                        print(f"  └─ [WARNING] Could not retrieve current_user(): {user_error}")

                    raise

                # ================================================================
                # CLEAN UP DELETED ROWS FROM AUDIT TABLE (REINSERTS)
                # ================================================================
                log_state("CLEANUP - REMOVE REINSERTED ROWS FROM AUDIT TABLE", deleted_rows_table)
                loggerObject.info(f"Deleting already deleted rows from {deleted_rows_table}.")
                print(f"  └─ Removing reinserted rows from {deleted_rows_table}...")

                cleanup_merge_on = " AND ".join([f"trim(upper(t.`{col}`)) = trim(upper(d.`{col}`))" for col in pk_cols])

                try:
                    spark.sql(f"""
                        MERGE INTO {deleted_rows_table} t
                        USING {target_table} d
                        ON {cleanup_merge_on}
WHEN MATCHED THEN
                        DELETE
                    """)

                    log_state("AFTER CLEANUP - SUCCESS", deleted_rows_table)
                    check_table_visibility(deleted_rows_table, "POST-CLEANUP")

                except Exception as cleanup_error:
                    log_state("AFTER CLEANUP - FAILED", deleted_rows_table)
                    print(f"  └─ [ERROR] Failed during cleanup: {str(cleanup_error)}")
                    print(f"  └─ Traceback: {traceback.format_exc()}")

                    try:
                        error_user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
                        print(f"  └─ Current user at error time: {error_user}")
                    except Exception as user_error:
                        print(f"  └─ [WARNING] Could not retrieve current_user(): {user_error}")

                    raise

        # ====================================================================
        # FUNCTION EXIT - SUCCESS
        # ====================================================================
        log_state("FUNCTION EXIT - SUCCESS", target_table)
        print("=" * 80)

    except Exception as e:
        # ====================================================================
        # FUNCTION EXIT - FAILURE
        # ====================================================================
        log_state("FUNCTION EXIT - FAILURE", target_table)
        print(f"[ERROR] Error during merge_into_stage: {str(e)}")
        print(f"[ERROR] Full traceback:\n{traceback.format_exc()}")

        try:
            final_user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
            print(f"[ERROR] Current user at final error: {final_user}")
        except Exception as user_error:
            print(f"[ERROR] Could not retrieve current_user(): {user_error}")

        print("=" * 80)
        raise